# Original group notebook (archived as-is). Data paths point to the team's private Google Drive; see the repository README to run the refactored code.

# We have a README file to explain all our main files.

## TF-IDF Leave-2-Out Evaluation for Top 5 Most Active Users

### Step 1: Load necessary libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix
from random import sample

### Step 2: Load the dataset

In [ ]:
df = pd.read_csv('user_game_with_all_fields.csv')

### Step 3: Combine text features (genre + description)

In [ ]:
df['text'] = df['genre'].fillna('') + ' ' + df['description'].fillna('')

### Step 4: Drop duplicate games based on app_id

In [ ]:
games_df = df.drop_duplicates(subset='app_id').reset_index(drop=True)

### Step 5: Extract TF-IDF features from text

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(games_df['text'])

### Step 6: Normalize hours played and encode as sparse matrix

In [ ]:
scaler = StandardScaler()
hours_scaled = scaler.fit_transform(games_df[['hours']].fillna(0))
hours_sparse = csr_matrix(hours_scaled)

### Step 7: Encode is_recommended as binary sparse feature

In [ ]:
is_rec = games_df['is_recommended'].fillna(False).astype(int).values.reshape(-1, 1)
is_rec_sparse = csr_matrix(is_rec)

### Step 8: Combine all features into one matrix

In [ ]:
full_feature_matrix = hstack([tfidf_matrix, hours_sparse, is_rec_sparse])

### Step 9: Build mapping between app_id and matrix index

In [ ]:
app_id_to_index = {app_id: idx for idx, app_id in enumerate(games_df['app_id'])}

### Step 10: Define function to recommend games for a user

In [ ]:
def recommend_games_for_user(user_id, top_n=5):
    user_played = df[df['user_id'] == user_id]['app_id'].tolist()
    played_indices = [app_id_to_index[app] for app in user_played if app in app_id_to_index]
    if not played_indices:
        return []

    user_vector = np.asarray(full_feature_matrix[played_indices].mean(axis=0))
    similarity_scores = cosine_similarity(user_vector.reshape(1, -1), full_feature_matrix).flatten()

    for idx in played_indices:
        similarity_scores[idx] = 0

    top_indices = similarity_scores.argsort()[::-1][:top_n]
    result = games_df.iloc[top_indices][['app_id']].copy()
    result['similarity'] = similarity_scores[top_indices]
    result['user_id'] = user_id
    return result[['user_id', 'app_id', 'similarity']]

### Step 11: Get top 5 users who played the most games

In [ ]:
top_users = df['user_id'].value_counts().head(5).index.tolist()

In [ ]:
print("========== TF-IDF Leave-2-Out Evaluation ==========")
all_hits = 0
for user_id in top_users:
    user_games = df[df['user_id'] == user_id]['app_id'].unique().tolist()

    if len(user_games) < 8:
        print(f"User {user_id}: skipped (only played {len(user_games)} games)")
        continue

    held_out = sample(user_games, 2)
    train_games = [g for g in user_games if g not in held_out]
    train_indices = [app_id_to_index[g] for g in train_games if g in app_id_to_index]

    if not train_indices:
        print(f"User {user_id}: no games found in TF-IDF index.")
        continue

    # generate user vector from played games
    user_vec = np.asarray(tfidf_matrix[train_indices].mean(axis=0))
    sim_scores = cosine_similarity(user_vec.reshape(1, -1), tfidf_matrix).flatten()

    # exclude all played games
    played_idx = [app_id_to_index[g] for g in user_games if g in app_id_to_index]
    for idx in played_idx:
        sim_scores[idx] = 0

    top_idx = sim_scores.argsort()[::-1][:5]
    top_apps = games_df.iloc[top_idx]['app_id'].tolist()
    all_hits += len(set(held_out) & set(top_apps))

    # print results
    hits = [g for g in held_out if g in top_apps]
    print(f"User {user_id}")
    print(f"  Played: {len(user_games)} games")
    print(f"  Held-out: {held_out}")
    print(f"  Recommended: {top_apps}")
    print(f"  Hits: {hits} → Hit@5 = {len(hits)}/2")
    print("---------------------------------------------------")

#
print("\n========== Overall Recommendation Results ==========")
print(f"Total Hits: {all_hits} out of {len(top_users) * 2} (Hit@5 = {all_hits / (len(top_users) * 2):.2f})")
print("===================================================")

========== TF-IDF Leave-2-Out Evaluation ==========
User 11762249
  Played: 16 games
  Held-out: [346110, 518790]
  Recommended: [251570, 261550, 976730, 1599340, 1172470]
  Hits: [] → Hit@5 = 0/2
---------------------------------------------------
User 5361677
  Played: 14 games
  Held-out: [1850570, 1313140]
  Recommended: [1817070, 368260, 1599340, 306130, 261550]
  Hits: [] → Hit@5 = 0/2
---------------------------------------------------
User 4947597
  Played: 14 games
  Held-out: [1172620, 413150]
  Recommended: [261550, 1599340, 306130, 440, 892970]
  Hits: [] → Hit@5 = 0/2
---------------------------------------------------
User 5188917
  Played: 13 games
  Held-out: [962130, 960090]
  Recommended: [1599340, 892970, 251570, 570940, 440]
  Hits: [] → Hit@5 = 0/2
---------------------------------------------------
User 8865664
  Played: 12 games
  Held-out: [275850, 304390]
  Recommended: [377160, 374320, 251570, 22380, 233860]
  Hits: [] → Hit@5 = 0/2
---------------------------

### Why We Switched to the "10-choice 2-hit" Evaluation Method

In our initial evaluation, we attempted to measure recommendation quality by checking whether 2 held-out games (from a user's history) appeared in the top-5 recommendations drawn from a candidate pool of tens of thousands of games. However, this resulted in extremely low hit rates due to the vast size of the candidate set, making the metric almost meaningless.

To address this, we adopted a more practical **10-choice 2-hit** strategy. For each user, we randomly sample 8 games they have never played and combine them with 2 held-out games from their history to form a small pool of 10 candidate games. We then compute similarity scores for these 10 candidates and rank them. Finally, we evaluate whether the 2 held-out games appear in the top-5, giving us a more realistic and informative Hit@5 or Recall@5 metric under controlled conditions.


In [ ]:
print("========== TF-IDF Leave-2-Out (Sampled Candidate Set) Evaluation ==========")
all_hits = 0
for user_id in top_users:
    user_games = df[df['user_id'] == user_id]['app_id'].unique().tolist()

    if len(user_games) < 8:
        print(f"User {user_id}: skipped (only played {len(user_games)} games)")
        continue

    held_out = sample(user_games, 2)
    train_games = [g for g in user_games if g not in held_out]
    train_indices = [app_id_to_index[g] for g in train_games if g in app_id_to_index]

    if not train_indices:
        print(f"User {user_id}: no games found in TF-IDF index.")
        continue

    # construct candidate pool
    all_app_ids = set(games_df['app_id'].unique())
    unplayed = list(all_app_ids - set(user_games))
    candidate_pool = sample(unplayed, 8) + held_out

    # transform to candidate indices
    candidate_indices = [app_id_to_index[g] for g in candidate_pool if g in app_id_to_index]
    if len(candidate_indices) < 10:
        print(f"User {user_id}: some candidate games missing from index.")
        continue

    # generate user vector from played games
    user_vec = np.asarray(tfidf_matrix[train_indices].mean(axis=0))
    sim_scores = cosine_similarity(user_vec.reshape(1, -1), tfidf_matrix[candidate_indices]).flatten()

    top_k_idx = np.argsort(sim_scores)[::-1][:5]
    top_apps = [candidate_pool[i] for i in top_k_idx]
    all_hits += len(set(held_out) & set(top_apps))
    # print results
    hits = [g for g in held_out if g in top_apps]
    print(f"User {user_id}")
    print(f"  Held-out (ground truth): {held_out}")
    print(f"  Top-5 recommended: {top_apps}")
    print(f"  Hits: {hits} → Hit@5 = {len(hits)}/2")
    print("---------------------------------------------------")

# 查看总的推荐结果
print("\n========== Overall Recommendation Results ==========")
print(f"Total Hits: {all_hits} out of {len(top_users) * 2} (Hit@5 = {all_hits / (len(top_users) * 2):.2f})")


========== TF-IDF Leave-2-Out (Sampled Candidate Set) Evaluation ==========
User 11762249
  Held-out (ground truth): [440900, 581320]
  Top-5 recommended: [440900, 1184370, 581320, 1824220, 2167580]
  Hits: [440900, 581320] → Hit@5 = 2/2
---------------------------------------------------
User 5361677
  Held-out (ground truth): [1824220, 1817190]
  Top-5 recommended: [1817190, 233860, 440, 858820, 1824220]
  Hits: [1824220, 1817190] → Hit@5 = 2/2
---------------------------------------------------
User 4947597
  Held-out (ground truth): [629730, 250900]
  Top-5 recommended: [261550, 1259420, 1286830, 960090, 815370]
  Hits: [] → Hit@5 = 0/2
---------------------------------------------------
User 5188917
  Held-out (ground truth): [1172620, 367520]
  Top-5 recommended: [1527950, 367520, 1172620, 107410, 236390]
  Hits: [1172620, 367520] → Hit@5 = 2/2
---------------------------------------------------
User 8865664
  Held-out (ground truth): [1332010, 493340]
  Top-5 recommended: [85882